# 04 - Merge Cleaned Laptop Data

Notebook này đưa hai nguồn đã cleaning (`Chợ Tốt` listing data và `Websach` product/shop data) về cùng canonical schema, append thành modeling population, kiểm tra source bias/leakage, và lưu output cho notebook feature engineering tiếp theo.

Notebook này không train model, không fit encoder/scaler, không one-hot encode, và không thực hiện feature engineering chính thức.

## 1. Setup

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != "Laptop-Price-Prediction":
    candidates = [p for p in PROJECT_ROOT.parents if p.name == "Laptop-Price-Prediction"]
    PROJECT_ROOT = candidates[0] if candidates else PROJECT_ROOT

CHOTOT_PATH = PROJECT_ROOT / "data" / "intern" / "chotot_laptop_cleaned_full.csv"
WEBSACH_PATH = PROJECT_ROOT / "data" / "intern" / "websach_cleaned_full.csv"

MERGED_PATH = PROJECT_ROOT / "data" / "intern" / "laptop_merged_cleaned.csv"
REPORT_PATH = PROJECT_ROOT / "docs" / "laptop_merged_report.csv"
CONFIG_PATH = PROJECT_ROOT / "docs" / "laptop_merge_config.json"

required_input_paths = {"chotot_cleaned": CHOTOT_PATH, "websach_cleaned": WEBSACH_PATH}
for name, path in required_input_paths.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing required input file `{name}` at: {path}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CHOTOT_PATH:", CHOTOT_PATH)
print("WEBSACH_PATH:", WEBSACH_PATH)
print("MERGED_PATH:", MERGED_PATH)
print("REPORT_PATH:", REPORT_PATH)
print("CONFIG_PATH:", CONFIG_PATH)

PROJECT_ROOT: Y:\Python\Laptop-Price-Prediction
CHOTOT_PATH: Y:\Python\Laptop-Price-Prediction\data\intern\chotot_laptop_cleaned_full.csv
WEBSACH_PATH: Y:\Python\Laptop-Price-Prediction\data\intern\websach_cleaned_full.csv
MERGED_PATH: Y:\Python\Laptop-Price-Prediction\data\intern\laptop_merged_cleaned.csv
REPORT_PATH: Y:\Python\Laptop-Price-Prediction\docs\laptop_merged_report.csv
CONFIG_PATH: Y:\Python\Laptop-Price-Prediction\docs\laptop_merge_config.json


In [2]:
def print_input_profile(name: str, df: pd.DataFrame) -> None:
    print(f"\n{'=' * 90}\n{name}\n{'=' * 90}")
    print("Shape:", df.shape)
    print("\nColumns:")
    print(pd.Series(df.columns, name="column").to_string(index=False))
    print("\nDtypes:")
    print(df.dtypes.to_string())
    print("\nFirst 3 rows:")
    display(df.head(3))
    print("\nMissing summary:")
    missing = (
        pd.DataFrame({"missing_count": df.isna().sum(), "missing_rate": df.isna().mean()})
        .sort_values(["missing_rate", "missing_count"], ascending=False)
    )
    display(missing)

chotot_raw = pd.read_csv(CHOTOT_PATH)
websach_raw = pd.read_csv(WEBSACH_PATH)

print_input_profile("Chợ Tốt cleaned input", chotot_raw)
print_input_profile("Websach cleaned input", websach_raw)


Chợ Tốt cleaned input
Shape: (5866, 64)

Columns:
                              url
                            price
                            title
                             Hãng
                         Dòng máy
                       Tình trạng
              Chính sách bảo hành
                 Kích cỡ màn hình
                      Bộ vi xử lý
                              RAM
                    Card màn hình
                           Ổ cứng
                          Xuất xứ
                      Loại ổ cứng
                           _price
                 is_price_missing
                 is_price_outlier
                    price_segment
                     origin_clean
        origin_missing_or_unknown
                      brand_clean
                 brand_from_title
             brand_mismatch_title
                      model_clean
                 model_from_title
                  model_recovered
                          _ram_gb
                      _storage_

,url,price,title,Hãng,Dòng máy,Tình trạng,Chính sách bảo hành,Kích cỡ màn hình,Bộ vi xử lý,RAM,Card màn hình,Ổ cứng,Xuất xứ,Loại ổ cứng,_price,is_price_missing,is_price_outlier,price_segment,origin_clean,origin_missing_or_unknown,brand_clean,brand_from_title,brand_mismatch_title,model_clean,model_from_title,model_recovered,_ram_gb,_storage_gb,_screen_size_inch,_title_ram_gb,_title_storage_gb,_title_cpu,_title_screen_size_inch,_ram_gb_filled,_storage_gb_filled,_screen_size_inch_filled,ram_missing,storage_capacity_missing,screen_missing,ram_suspicious,storage_missing_all,cpu_missing,cpu_brand,cpu_tier,gpu_missing,gpu_tier,potential_dedicated_gpu,warranty_months,warranty_status,has_warranty,storage_type_clean,was_storage_type_imputed,condition_clean,new_low_price,repair_mismatch,condition_warranty_inconsistent,brand_grouped,model_grouped,brand_is_rare,model_is_rare,_ram_gb_original_before_qc,_ram_gb_filled_original_before_qc,_title_ram_gb_original_before_qc,ram_gb_original_before_qc
0,https://www.chotot.com/mua-ban-quan-tan-binh-t...,9.990.000 đ,Laptop 2in1 Dell 7420 I7 1185G7/16G/256G,Dell,Latitude,Đã sử dụng (chưa sửa chữa),>12 tháng,13 - 14.9 inch,Intel Core i7,16 GB,Onboard,256 GB,Mỹ,SSD,"9,990,000.0000",False,False,mid,Mỹ,False,Dell,Dell,False,Latitude,Unknown,Latitude,16.0000,256.0000,13.9500,16.0000,NaN,Unknown,NaN,16.0000,256.0000,13.9500,False,False,False,False,False,False,Intel,Upper-mid,False,Integrated - Intel,False,13.0000,Active,True,SSD,False,Đã sử dụng (chưa sửa chữa),False,False,False,Dell,Latitude,False,False,16.0000,16.0000,16.0000,16.0000
1,https://www.chotot.com/mua-ban-thanh-pho-thu-d...,4.500.000 đ,Acer Aspire A315-58 i3-1115G4,Acer,Aspire A3,Đã sử dụng (chưa sửa chữa),Bảo hành hãng,15 - 16.9 inch,Intel Core i3,8 GB,Onboard,< 128 GB,Đang cập nhật,SSD,"4,500,000.0000",False,False,low,Unknown,True,Acer,Acer,False,Aspire A3,Aspire,Aspire A3,8.0000,127.0000,15.9500,NaN,NaN,Unknown,NaN,8.0000,127.0000,15.9500,False,False,False,False,False,False,Intel,Entry,False,Integrated - Intel,False,NaN,Manufacturer,True,SSD,False,Đã sử dụng (chưa sửa chữa),False,False,False,Acer,Other,False,True,8.0000,8.0000,NaN,8.0000
2,https://www.chotot.com/mua-ban-quan-ha-dong-ha...,3.500.000 đ,Thanh lý Laptop HP Elite x2 1012G1 cảm ứng 2in1,HP,Elite X2,Đã sử dụng (chưa sửa chữa),Bảo hành hãng,11 - 12.9 inch,Intel Core i7,8 GB,Onboard,512 GB,Đang cập nhật,SSD,"3,500,000.0000",False,False,low,Unknown,True,HP,HP,False,Elite X2,Unknown,Elite X2,8.0000,512.0000,11.9500,NaN,NaN,Unknown,NaN,8.0000,512.0000,11.9500,False,False,False,False,False,False,Intel,Upper-mid,False,Integrated - Intel,False,NaN,Manufacturer,True,SSD,False,Đã sử dụng (chưa sửa chữa),False,False,False,HP,Other,False,True,8.0000,8.0000,NaN,8.0000



Missing summary:


,missing_count,missing_rate
_title_screen_size_inch,4111,0.7008
warranty_months,3392,0.5782
_title_storage_gb,2730,0.4654
_title_ram_gb,1638,0.2792
_title_ram_gb_original_before_qc,1621,0.2763
Card màn hình,1421,0.2422
Kích cỡ màn hình,1037,0.1768
_screen_size_inch,1037,0.1768
_screen_size_inch_filled,998,0.1701
Loại ổ cứng,726,0.1238



Websach cleaned input
Shape: (4384, 67)

Columns:
                Hãng sản xuất
                Công nghệ CPU
                     Loại CPU
               Dung lượng RAM
                     Loại RAM
                  Loại ổ cứng
       Dung lượng ổ cứng (GB)
           Đồ họa đã làm sạch
            Kích thước (inch)
                 shop_1_price
                  shop_1_name
                 shop_2_price
                  shop_2_name
                 shop_3_price
                  shop_3_name
  shop_1_price_domain_outlier
          shop_1_price_domain
           shop_1_price_clean
  shop_2_price_domain_outlier
          shop_2_price_domain
           shop_2_price_clean
  shop_3_price_domain_outlier
          shop_3_price_domain
           shop_3_price_clean
      price_row_median_domain
shop_1_price_relative_outlier
shop_2_price_relative_outlier
shop_3_price_relative_outlier
                 n_prices_raw
               n_prices_clean
                 price_median
              price

,Hãng sản xuất,Công nghệ CPU,Loại CPU,Dung lượng RAM,Loại RAM,Loại ổ cứng,Dung lượng ổ cứng (GB),Đồ họa đã làm sạch,Kích thước (inch),shop_1_price,shop_1_name,shop_2_price,shop_2_name,shop_3_price,shop_3_name,shop_1_price_domain_outlier,shop_1_price_domain,shop_1_price_clean,shop_2_price_domain_outlier,shop_2_price_domain,shop_2_price_clean,shop_3_price_domain_outlier,shop_3_price_domain,shop_3_price_clean,price_row_median_domain,shop_1_price_relative_outlier,shop_2_price_relative_outlier,shop_3_price_relative_outlier,n_prices_raw,n_prices_clean,price_median,price_min_clean,price_max_clean,price_spread_clean_pct,flag_price_spread_warn,flag_price_spread_critical,log_price_median,price_segment,brand_clean,brand_grouped,brand_is_rare,screen_size_clean,screen_size_outlier,screen_size_group,cpu_brand,cpu_tier,cpu_model_clean,ram_gb_clean,ram_capacity_outlier,ram_tier_clean,ram_type_clean,storage_type_clean,storage_gb_clean,storage_gb_outlier,storage_tier_clean,gpu_brand,gpu_tier_v2,product_id,spec_key,is_soft_duplicate_spec,high_end_config_flag,brand_screen_cell_n,low_count_brand_screen_cell,cpu_ram_cell_n,gpu_ram_cell_n,low_count_cpu_ram_cell,low_count_gpu_ram_cell
0,Acer,Intel Core i5,8265U,8.0000,DDR4,SSD,512.0000,Intel UHD Graphics,14.0000,14950000,baochau.vn,15950000,quangmai.net,"15,990,000.0000",compro.com.vn,False,"14,950,000.0000","14,950,000.0000",False,15950000,"15,950,000.0000",False,"15,990,000.0000","15,990,000.0000","15,950,000.0000",False,False,False,3,3,"15,950,000.0000","14,950,000.0000","15,990,000.0000",0.0652,False,False,16.5850,mid,Acer,Acer,False,14.0000,False,14-14.9in,Intel,Mid-range,8265U,8.0000,False,<=8GB,DDR4,SSD,512.0000,False,512GB,Intel,Intel Integrated,0,Acer|8265U|8.0|512.0|14.0,False,False,84,False,1774,1101,False,False
1,Lg,Intel Core i7,1165G7,16.0000,LPDDR4x,SSD,512.0000,Intel Iris Xe,17.0000,32640000,tiki.vn,47500000,tinhocsangtao.vn,"35,190,000.0000",mixicomputer.vn,False,"32,640,000.0000","32,640,000.0000",False,47500000,"47,500,000.0000",False,"35,190,000.0000","35,190,000.0000","35,190,000.0000",False,False,False,3,3,"35,190,000.0000","32,640,000.0000","47,500,000.0000",0.4223,True,False,17.3763,high,LG,LG,False,17.0000,False,>=17in,Intel,Upper-mid,1165G7,16.0000,False,16GB,LPDDR4X,SSD,512.0000,False,512GB,Intel,Intel Integrated,1,LG|1165G7|16.0|512.0|17.0,False,False,6,True,1297,753,False,False
2,Lenovo,AMD Ryzen 7,8845HS,16.0000,DDR5,SSD,512.0000,RTX 4060,16.0000,30900000,congnghesgsaigon.com,31490000,tnc.com.vn,"33,490,000.0000",tymo.vn,False,"30,900,000.0000","30,900,000.0000",False,31490000,"31,490,000.0000",False,"33,490,000.0000","33,490,000.0000","31,490,000.0000",False,False,False,3,3,"31,490,000.0000","30,900,000.0000","33,490,000.0000",0.0822,False,False,17.2652,high,Lenovo,Lenovo,False,16.0000,False,16-16.9in,AMD,Upper-mid,8845HS,16.0000,False,16GB,DDR5,SSD,512.0000,False,512GB,NVIDIA,RTX 4000,2,Lenovo|8845HS|16.0|512.0|16.0,False,True,184,False,252,180,False,False



Missing summary:


,missing_count,missing_rate
Loại RAM,318,0.0725
Dung lượng ổ cứng (GB),225,0.0513
storage_gb_clean,225,0.0513
Loại ổ cứng,219,0.0500
Hãng sản xuất,180,0.0411
screen_size_clean,96,0.0219
Công nghệ CPU,94,0.0214
Loại CPU,69,0.0157
shop_1_price_clean,33,0.0075
ram_gb_clean,23,0.0052


## 2. Define Canonical Schema

In [3]:
CANONICAL_COLUMNS = [
    "source", "source_row_id", "merged_row_id", "url", "title",
    "target_price", "log_target_price", "target_price_source",
    "target_price_is_outlier", "target_price_missing",
    "brand_clean", "brand_grouped", "brand_is_rare",
    "model_clean", "model_grouped", "model_is_rare",
    "ram_gb", "storage_gb", "screen_size_inch",
    "cpu_name_raw", "cpu_type", "cpu_model_clean",
    "cpu_brand", "cpu_tier", "gpu_tier", "gpu_tier_v2", "storage_type_clean",
    "condition_clean", "warranty_months", "warranty_status", "has_warranty", "origin_clean",
    "ram_missing", "storage_missing", "screen_missing", "cpu_missing", "gpu_missing",
    "ram_suspicious", "storage_suspicious", "screen_suspicious",
    "potential_dedicated_gpu", "repair_mismatch",
    "price_spread_clean_pct", "flag_price_spread_warn", "flag_price_spread_critical",
    "is_soft_duplicate_spec", "merged_spec_key", "is_cross_source_soft_duplicate",
]

LABEL_COLUMNS = ["target_price", "log_target_price"]
AUDIT_COLUMNS = [
    "source", "source_row_id", "merged_row_id", "url", "title",
    "target_price_source", "target_price_is_outlier", "target_price_missing",
    "merged_spec_key", "is_cross_source_soft_duplicate",
]

assert len(CANONICAL_COLUMNS) == len(set(CANONICAL_COLUMNS)), "Canonical columns must be unique."
print(f"Canonical column count: {len(CANONICAL_COLUMNS)}")
print("Label columns:", LABEL_COLUMNS)
print("Audit columns:", AUDIT_COLUMNS)

Canonical column count: 48
Label columns: ['target_price', 'log_target_price']
Audit columns: ['source', 'source_row_id', 'merged_row_id', 'url', 'title', 'target_price_source', 'target_price_is_outlier', 'target_price_missing', 'merged_spec_key', 'is_cross_source_soft_duplicate']


## 3. Define Source Mappings

In [4]:
CHOTOT_MAPPING = {
    "_price": "target_price",
    "is_price_outlier": "target_price_is_outlier",
    "is_price_missing": "target_price_missing",
    "brand_clean": "brand_clean",
    "brand_grouped": "brand_grouped",
    "brand_is_rare": "brand_is_rare",
    "model_clean": "model_clean",
    "model_grouped": "model_grouped",
    "model_is_rare": "model_is_rare",
    "_ram_gb_filled": "ram_gb",
    "_storage_gb_filled": "storage_gb",
    "_screen_size_inch_filled": "screen_size_inch",
    "_title_cpu": "cpu_name_raw",
    "cpu_brand": "cpu_brand",
    "cpu_tier": "cpu_tier",
    "gpu_tier": "gpu_tier",
    "storage_type_clean": "storage_type_clean",
    "condition_clean": "condition_clean",
    "warranty_months": "warranty_months",
    "warranty_status": "warranty_status",
    "has_warranty": "has_warranty",
    "origin_clean": "origin_clean",
    "ram_missing": "ram_missing",
    "storage_capacity_missing": "storage_missing",
    "screen_missing": "screen_missing",
    "cpu_missing": "cpu_missing",
    "gpu_missing": "gpu_missing",
    "ram_suspicious": "ram_suspicious",
    "potential_dedicated_gpu": "potential_dedicated_gpu",
    "repair_mismatch": "repair_mismatch",
    "url": "url",
    "title": "title",
}

WEBSACH_MAPPING = {
    "price_median": "target_price",
    "log_price_median": "log_target_price",
    "brand_clean": "brand_clean",
    "brand_grouped": "brand_grouped",
    "brand_is_rare": "brand_is_rare",
    "ram_gb_clean": "ram_gb",
    "storage_gb_clean": "storage_gb",
    "screen_size_clean": "screen_size_inch",
    "Công nghệ CPU": "cpu_name_raw",
    "Loại CPU": "cpu_type",
    "cpu_model_clean": "cpu_model_clean",
    "cpu_brand": "cpu_brand",
    "cpu_tier": "cpu_tier",
    "gpu_tier_v2": "gpu_tier",
    "gpu_tier_v2_raw": "gpu_tier_v2",
    "storage_type_clean": "storage_type_clean",
    "price_spread_clean_pct": "price_spread_clean_pct",
    "flag_price_spread_warn": "flag_price_spread_warn",
    "flag_price_spread_critical": "flag_price_spread_critical",
    "is_soft_duplicate_spec": "is_soft_duplicate_spec",
}

possible_websach_title_cols = ["title", "Tên sản phẩm", "product_name", "name", "Tên laptop"]
for col in possible_websach_title_cols:
    if col in websach_raw.columns:
        WEBSACH_MAPPING[col] = "title"
        print(f"Mapped Websach title from `{col}`.")
        break

possible_websach_url_cols = ["url", "product_url", "link", "product_link", "URL"]
for col in possible_websach_url_cols:
    if col in websach_raw.columns:
        WEBSACH_MAPPING[col] = "url"
        print(f"Mapped Websach URL from `{col}`.")
        break

print("Chợ Tốt mapped columns:", len(CHOTOT_MAPPING))
print("Websach mapped columns:", len(WEBSACH_MAPPING))

Chợ Tốt mapped columns: 32
Websach mapped columns: 20


## 4. Define Defaults

In [5]:
CHOTOT_DEFAULTS = {
    "source": "chotot",
    "target_price_source": "listing_price",
    "price_spread_clean_pct": np.nan,
    "flag_price_spread_warn": False,
    "flag_price_spread_critical": False,
    "storage_suspicious": False,
    "screen_suspicious": False,
    "is_soft_duplicate_spec": False,
}

WEBSACH_DEFAULTS = {
    "source": "websach",
    "target_price_source": "shop_price_median",
    "target_price_is_outlier": False,
    "model_clean": "Unknown",
    "model_grouped": "Unknown",
    "model_is_rare": np.nan,
    "condition_clean": "Unknown",
    "warranty_months": np.nan,
    "warranty_status": "Unknown",
    "has_warranty": np.nan,
    "origin_clean": "Unknown",
    "url": pd.NA,
    "title": pd.NA,
    "ram_suspicious": False,
    "storage_suspicious": False,
    "screen_suspicious": False,
    "potential_dedicated_gpu": False,
    "repair_mismatch": False,
}

## 5. Helper Functions

In [6]:
def safe_log1p_price(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce")
    values = values.mask(values <= 0)
    return np.log1p(values)


def to_bool_series(s: pd.Series, default: bool = False, preserve_na: bool = False) -> pd.Series:
    if pd.api.types.is_bool_dtype(s):
        return s.astype("boolean") if preserve_na else s.fillna(default).astype(bool)

    normalized = s.astype("string").str.strip().str.lower()
    mapped = normalized.map({
        "true": True, "false": False, "1": True, "0": False,
        "1.0": True, "0.0": False, "yes": True, "no": False,
        "y": True, "n": False, "t": True, "f": False,
        "nan": pd.NA if preserve_na else default,
        "<na>": pd.NA if preserve_na else default,
        "none": pd.NA if preserve_na else default,
        "": pd.NA if preserve_na else default,
    })
    return mapped.astype("boolean") if preserve_na else mapped.fillna(default).astype(bool)


def coerce_boolean_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    strict_bool_cols = [
        "target_price_is_outlier", "target_price_missing",
        "ram_missing", "storage_missing", "screen_missing", "cpu_missing", "gpu_missing",
        "ram_suspicious", "storage_suspicious", "screen_suspicious",
        "potential_dedicated_gpu", "repair_mismatch",
        "flag_price_spread_warn", "flag_price_spread_critical",
        "is_soft_duplicate_spec", "is_cross_source_soft_duplicate",
    ]
    nullable_bool_cols = ["brand_is_rare", "model_is_rare", "has_warranty"]
    for col in strict_bool_cols:
        if col in out.columns:
            out[col] = to_bool_series(out[col], default=False, preserve_na=False)
    for col in nullable_bool_cols:
        if col in out.columns:
            out[col] = to_bool_series(out[col], default=False, preserve_na=True)
    return out


def missing_like_rate(s: pd.Series) -> float:
    if pd.api.types.is_numeric_dtype(s):
        return float(s.isna().mean())
    normalized = s.astype("string").str.strip().str.lower()
    missing_like = s.isna() | normalized.isin(["unknown", "missing", "nan", "<na>", "none", ""])
    return float(missing_like.mean())


def missing_like_count(s: pd.Series) -> int:
    if pd.api.types.is_numeric_dtype(s):
        return int(s.isna().sum())
    normalized = s.astype("string").str.strip().str.lower()
    missing_like = s.isna() | normalized.isin(["unknown", "missing", "nan", "<na>", "none", ""])
    return int(missing_like.sum())


def canonicalize_source(df, mapping, canonical_columns, defaults, source_name):
    out = df.copy()
    out["source_row_id"] = out.index
    missing_mapped_cols = sorted(set(mapping) - set(out.columns))
    if missing_mapped_cols:
        print(f"WARNING: {source_name} mapped source columns not found: {missing_mapped_cols}")

    existing_mapping = {src: dst for src, dst in mapping.items() if src in out.columns}
    if "target_price" not in existing_mapping.values():
        raise ValueError(
            f"{source_name}: no source column was mapped to `target_price`. "
            "Check mapping keys and input schema."
        )

    out = out.rename(columns=existing_mapping)

    for col, value in defaults.items():
        if col == "source":
            out[col] = value
        elif col not in out.columns:
            out[col] = value
    for col in canonical_columns:
        if col not in out.columns:
            out[col] = np.nan

    out["source"] = defaults.get("source", source_name)
    out["target_price"] = pd.to_numeric(out["target_price"], errors="coerce")
    if out["target_price"].notna().sum() == 0:
        raise ValueError(f"{source_name}: `target_price` is entirely missing after mapping.")

    out["log_target_price"] = safe_log1p_price(out["target_price"])

    if source_name == "websach":
        out["target_price_missing"] = out["target_price"].isna()
        out["ram_missing"] = out["ram_gb"].isna()
        out["storage_missing"] = out["storage_gb"].isna()
        out["screen_missing"] = out["screen_size_inch"].isna()
        out["cpu_missing"] = out["cpu_brand"].isna() | out["cpu_brand"].eq("Unknown")
        out["gpu_missing"] = out["gpu_tier"].isna() | out["gpu_tier"].eq("Unknown")
    elif source_name == "chotot":
        fallback_flags = {
            "target_price_missing": out["target_price"].isna(),
            "ram_missing": out["ram_gb"].isna(),
            "storage_missing": out["storage_gb"].isna(),
            "screen_missing": out["screen_size_inch"].isna(),
            "cpu_missing": out["cpu_brand"].isna() | out["cpu_brand"].eq("Unknown"),
            "gpu_missing": out["gpu_tier"].isna() | out["gpu_tier"].isin(["Missing", "Unknown"]),
        }
        for col, value in fallback_flags.items():
            if col not in existing_mapping.values() or out[col].isna().all():
                out[col] = value

    return coerce_boolean_columns(out[canonical_columns].copy())


def normalize_key_value(value, col: str) -> str:
    if pd.isna(value):
        return "Unknown"
    if col in {"ram_gb", "storage_gb"}:
        try:
            return str(int(round(float(value))))
        except (TypeError, ValueError):
            return "Unknown"
    if col == "screen_size_inch":
        try:
            return f"{float(value):.1f}"
        except (TypeError, ValueError):
            return "Unknown"
    text = str(value).strip()
    return text if text and text.lower() not in {"nan", "<na>", "none"} else "Unknown"


def build_spec_key(df: pd.DataFrame, spec_key_cols: list[str]) -> pd.Series:
    key_parts = []
    for col in spec_key_cols:
        if col not in df.columns:
            key_parts.append(pd.Series("Unknown", index=df.index))
        else:
            key_parts.append(df[col].apply(lambda x, c=col: normalize_key_value(x, c)))
    return pd.concat(key_parts, axis=1).agg("|".join, axis=1)


def count_by_source(df: pd.DataFrame, label: str) -> pd.DataFrame:
    return df.groupby("source", dropna=False).size().rename(label).reset_index()


def summarize_target_by_source(df: pd.DataFrame) -> pd.DataFrame:
    return df.groupby("source")["target_price"].agg(
        count="count", mean="mean", median="median", std="std", min="min",
        p05=lambda s: s.quantile(0.05), p25=lambda s: s.quantile(0.25),
        p75=lambda s: s.quantile(0.75), p95=lambda s: s.quantile(0.95), max="max",
    ).reset_index()


def to_jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, pd.DataFrame):
        return to_jsonable(obj.to_dict(orient="records"))
    if isinstance(obj, pd.Series):
        return to_jsonable(obj.to_dict())
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return None if np.isnan(obj) else float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if obj is pd.NA:
        return None
    if isinstance(obj, float) and np.isnan(obj):
        return None
    return obj

## 6. Canonicalize Both Sources

In [7]:
websach_for_merge = websach_raw.copy()
if "gpu_tier_v2" in websach_for_merge.columns:
    websach_for_merge["gpu_tier_v2_raw"] = websach_for_merge["gpu_tier_v2"]
else:
    websach_for_merge["gpu_tier_v2_raw"] = np.nan

chotot_can = canonicalize_source(chotot_raw, CHOTOT_MAPPING, CANONICAL_COLUMNS, CHOTOT_DEFAULTS, "chotot")
websach_can = canonicalize_source(websach_for_merge, WEBSACH_MAPPING, CANONICAL_COLUMNS, WEBSACH_DEFAULTS, "websach")

print("Chợ Tốt canonical shape:", chotot_can.shape)
print("Websach canonical shape:", websach_can.shape)

completeness = pd.concat(
    [chotot_can.notna().mean().rename("chotot_completeness"),
     websach_can.notna().mean().rename("websach_completeness")],
    axis=1,
)
display(completeness)

new_cols = [
    "cpu_name_raw",
    "cpu_type",
    "cpu_model_clean",
    "gpu_tier",
    "gpu_tier_v2",
]

print("New CPU/GPU columns availability by source:")
for name, df_can in {
    "chotot": chotot_can,
    "websach": websach_can,
}.items():
    print(f"\n=== {name.upper()} ===")
    display(
        df_can[new_cols]
        .isna()
        .mean()
        .rename("missing_rate")
        .reset_index()
        .rename(columns={"index": "column"})
    )

print("\nSample Chợ Tốt:")
display(
    chotot_can[
        ["source", "title", "cpu_name_raw", "cpu_type", "cpu_model_clean", "cpu_brand", "cpu_tier", "gpu_tier", "gpu_tier_v2"]
    ].head(10)
)

print("\nSample Websach:")
display(
    websach_can[
        ["source", "title", "cpu_name_raw", "cpu_type", "cpu_model_clean", "cpu_brand", "cpu_tier", "gpu_tier", "gpu_tier_v2"]
    ].head(10)
)

print("Target price summary before filtering:")
display(summarize_target_by_source(pd.concat([chotot_can, websach_can], ignore_index=True)))

Chợ Tốt canonical shape: (5866, 48)
Websach canonical shape: (4384, 48)


C:\Users\Duy\AppData\Local\Temp\ipykernel_10956\1271337414.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return mapped.astype("boolean") if preserve_na else mapped.fillna(default).astype(bool)
C:\Users\Duy\AppData\Local\Temp\ipykernel_10956\1271337414.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return mapped.astype("boolean") if preserve_na else mapped.fillna(default).astype(bool)


,chotot_completeness,websach_completeness
source,1.0000,1.0000
source_row_id,1.0000,1.0000
merged_row_id,0.0000,0.0000
url,1.0000,0.0000
title,1.0000,0.0000
target_price,0.9998,1.0000
log_target_price,0.9998,1.0000
target_price_source,1.0000,1.0000
target_price_is_outlier,1.0000,1.0000
target_price_missing,1.0000,1.0000


New CPU/GPU columns availability by source:

=== CHOTOT ===


,column,missing_rate
0,cpu_name_raw,0.0000
1,cpu_type,1.0000
2,cpu_model_clean,1.0000
3,gpu_tier,0.0000
4,gpu_tier_v2,1.0000


...

,source,title,cpu_name_raw,cpu_type,cpu_model_clean,cpu_brand,cpu_tier,gpu_tier,gpu_tier_v2
0,chotot,Laptop 2in1 Dell 7420 I7 1185G7/16G/256G,Unknown,NaN,NaN,Intel,Upper-mid,Integrated - Intel,NaN
1,chotot,Acer Aspire A315-58 i3-1115G4,Unknown,NaN,NaN,Intel,Entry,Integrated - Intel,NaN
2,chotot,Thanh lý Laptop HP Elite x2 1012G1 cảm ứng 2in1,Unknown,NaN,NaN,Intel,Upper-mid,Integrated - Intel,NaN
3,chotot,Dell Pro Rugged RB14250 Ultra 7-165U 32GB 512GB,Unknown,NaN,NaN,Intel,Other,Integrated - Intel,NaN
4,chotot,HP Spectre x360 13-ae015dx i7 cảm ứng gập 360 độ,Unknown,NaN,NaN,Unknown,Missing,Integrated - Intel,NaN
5,chotot,HP Elitebook 840 G5 i7 8650U 8GB/256GB,i7 8650u,NaN,NaN,Intel,Upper-mid,Integrated - Intel,NaN
6,chotot,Dell Latitude 7300 i5 13 inch 12GB/256GB,Unknown,NaN,NaN,Intel,Mid,Integrated - Intel,NaN
7,chotot,MacBook Air 2019 | Core i5 | 16GB | 256GB,Unknown,NaN,NaN,Intel,Mid,Missing,NaN
8,chotot,Acer Nitro Tiger 12700H RTX 3050 keng hãng 99%,Unknown,NaN,NaN,Intel,Upper-mid,Dedicated - Other/Entry,NaN
9,chotot,Apple MacBook Pro 2017 i5 ram 8/128g zin all 99%,Unknown,NaN,NaN,Intel,Mid,Missing,NaN



Sample Websach:


,source,title,cpu_name_raw,cpu_type,cpu_model_clean,cpu_brand,cpu_tier,gpu_tier,gpu_tier_v2
0,websach,<NA>,Intel Core i5,8265U,8265U,Intel,Mid-range,Intel Integrated,Intel Integrated
1,websach,<NA>,Intel Core i7,1165G7,1165G7,Intel,Upper-mid,Intel Integrated,Intel Integrated
2,websach,<NA>,AMD Ryzen 7,8845HS,8845HS,AMD,Upper-mid,RTX 4000,RTX 4000
3,websach,<NA>,AMD Ryzen 7,7840H,7840H,AMD,Upper-mid,RTX 4000,RTX 4000
4,websach,<NA>,Intel Core Ultra 7,255HX,255HX,Intel,Upper-mid,RTX 5000,RTX 5000
5,websach,<NA>,Intel Core i9,13980HX,13980HX,Intel,High-end,Unknown,Unknown
6,websach,<NA>,Intel Core Ultra 7,255H,255H,Intel,Upper-mid,Intel Integrated,Intel Integrated
7,websach,<NA>,Intel Core i3,1305U,1305U,Intel,Entry,Intel Integrated,Intel Integrated
8,websach,<NA>,Intel Core i7,13700H,13700H,Intel,Upper-mid,RTX 4000,RTX 4000
9,websach,<NA>,NaN,NaN,UNKNOWN,Unknown,Unknown,AMD Radeon,AMD Radeon


Target price summary before filtering:


,source,count,mean,median,std,min,p05,p25,p75,p95,max
0,chotot,5865,"9,915,058.3478","7,600,000.0000","8,364,111.0304","6,500.0000","1,500,000.0000","4,500,000.0000","12,990,000.0000","25,972,000.0000","99,900,000.0000"
1,websach,4384,"26,031,729.3296","21,190,000.0000","17,393,209.8723","3,470,000.0000","9,790,000.0000","15,200,000.0000","30,990,000.0000","57,567,650.0000","179,400,000.0000"


### Canonicalization Notes

Hai nguồn đã được ép về cùng schema, nhưng tỷ lệ hoàn thiện giữa các cột không nên được hiểu như feature quality cuối cùng. Một số cột `Unknown`/missing là khác biệt bản chất nguồn: Websach không có condition/warranty/origin như listing Chợ Tốt, nên các cột đó chủ yếu phục vụ audit sau merge.

## 7. Concatenate

In [8]:
merged = pd.concat([chotot_can, websach_can], ignore_index=True)
merged["merged_row_id"] = np.arange(len(merged))
merged = coerce_boolean_columns(merged)

assert merged["source"].notna().all(), "source must not be missing."
assert merged["source_row_id"].notna().all(), "source_row_id must not be missing."

print("Merged pre-filter shape:", merged.shape)
display(count_by_source(merged, "rows_before_filtering"))

Merged pre-filter shape: (10250, 48)


,source,rows_before_filtering
0,chotot,5866
1,websach,4384


## 8. Target Filtering

In [9]:
row_counts_before_filtering = count_by_source(merged, "rows_before_filtering")

filter_audit_frames = []
missing_price_mask = merged["target_price"].isna()
non_positive_mask = merged["target_price"].notna() & merged["target_price"].le(0)
chotot_outlier_mask = merged["source"].eq("chotot") & to_bool_series(merged["target_price_is_outlier"], default=False)

for reason, mask in {
    "target_price_missing": missing_price_mask,
    "target_price_non_positive": non_positive_mask,
    "chotot_target_price_outlier": chotot_outlier_mask,
}.items():
    counts = merged.loc[mask].groupby("source", dropna=False).size().rename("removed_rows").reset_index()
    counts["reason"] = reason
    filter_audit_frames.append(counts)

filter_audit = pd.concat(filter_audit_frames, ignore_index=True) if filter_audit_frames else pd.DataFrame()
remove_mask = missing_price_mask | non_positive_mask | chotot_outlier_mask
merged = merged.loc[~remove_mask].copy()
merged["target_price"] = pd.to_numeric(merged["target_price"], errors="coerce")
merged["log_target_price"] = np.log1p(merged["target_price"])
merged = coerce_boolean_columns(merged)

row_counts_after_filtering = count_by_source(merged, "rows_after_filtering")

print("Removed rows by source and reason:")
display(filter_audit)
print("Row counts before filtering:")
display(row_counts_before_filtering)
print("Row counts after filtering:")
display(row_counts_after_filtering)

Removed rows by source and reason:


,source,removed_rows,reason
0,chotot,1,target_price_missing
1,chotot,167,chotot_target_price_outlier


Row counts before filtering:


,source,rows_before_filtering
0,chotot,5866
1,websach,4384


Row counts after filtering:


,source,rows_after_filtering
0,chotot,5698
1,websach,4384


### Filtering Notes

Filter target chỉ loại giá thiếu, giá không dương, và outlier price đã flag từ Chợ Tốt. Các flag price spread của Websach được giữ lại làm diagnostics để notebook modeling có thể đánh giá riêng nhóm spread cao thay vì loại bỏ sớm.

## 9. Deduplication

In [10]:
exact_duplicate_subset = [col for col in CANONICAL_COLUMNS if col not in {"merged_row_id", "source_row_id"}]
exact_duplicate_mask = merged.duplicated(subset=exact_duplicate_subset, keep="first")
exact_duplicate_removed_count = int(exact_duplicate_mask.sum())
merged = merged.loc[~exact_duplicate_mask].copy()
print("Exact duplicate rows removed:", exact_duplicate_removed_count)

has_url = merged["url"].notna() & merged["url"].astype("string").str.strip().ne("")
url_duplicate_mask = pd.Series(False, index=merged.index)
url_duplicate_mask.loc[has_url] = merged.loc[has_url].duplicated(subset=["source", "url"], keep="first")
url_duplicate_removed_count = int(url_duplicate_mask.sum())
merged = merged.loc[~url_duplicate_mask].copy()
print("URL duplicate rows removed:", url_duplicate_removed_count)

SOFT_DUPLICATE_KEY_COLUMNS = [
    "brand_grouped", "model_grouped", "cpu_brand", "cpu_tier",
    "ram_gb", "storage_gb", "screen_size_inch", "gpu_tier",
]

merged["merged_spec_key"] = build_spec_key(merged, SOFT_DUPLICATE_KEY_COLUMNS)
source_nunique_by_key = merged.groupby("merged_spec_key")["source"].transform("nunique")
merged["is_cross_source_soft_duplicate"] = source_nunique_by_key.gt(1)

soft_duplicate_count = int(merged.duplicated(subset=["merged_spec_key"], keep=False).sum())
cross_source_soft_duplicate_count = int(merged["is_cross_source_soft_duplicate"].sum())

merged = merged.reset_index(drop=True)
merged["merged_row_id"] = np.arange(len(merged))
merged["target_price"] = pd.to_numeric(merged["target_price"], errors="coerce")
merged["log_target_price"] = np.log1p(merged["target_price"])
merged = coerce_boolean_columns(merged)
merged = merged[CANONICAL_COLUMNS]

dup_key_counts = (
    merged.groupby("merged_spec_key")
    .agg(rows=("merged_spec_key", "size"), sources=("source", lambda s: ", ".join(sorted(s.unique()))))
    .query("rows > 1")
    .sort_values("rows", ascending=False)
    .head(20)
    .reset_index()
)

print("Total soft duplicate rows:", soft_duplicate_count)
print("Cross-source soft duplicate rows:", cross_source_soft_duplicate_count)
print("Top 20 duplicated spec keys:")
display(dup_key_counts)
print("Post-dedup shape:", merged.shape)

Exact duplicate rows removed: 2
URL duplicate rows removed: 0
Total soft duplicate rows: 6882
Cross-source soft duplicate rows: 0
Top 20 duplicated spec keys:


,merged_spec_key,rows,sources
0,Dell|Latitude|Intel|Mid|8|256|13.9|Integrated ...,84,chotot
1,Dell|Latitude|Intel|Upper-mid|16|256|13.9|Inte...,61,chotot
2,Dell|Precision|Intel|Upper-mid|16|512|15.9|Ded...,58,chotot
3,Dell|Latitude|Intel|Upper-mid|8|256|13.9|Integ...,55,chotot
4,Dell|Other|Intel|Upper-mid|16|512|15.9|Dedicat...,50,chotot
5,Lenovo|Unknown|Intel|Upper-mid|16|512|14.0|Int...,46,websach
6,Dell|Latitude|Intel|Mid|16|256|13.9|Integrated...,40,chotot
7,Dell|Dòng Khác|Intel|Mid|8|256|13.9|Integrated...,40,chotot
8,Dell|Other|Intel|Mid|8|256|13.9|Integrated - I...,39,chotot
9,ASUS|Unknown|Intel|Mid-range|8|512|14.0|Intel ...,34,websach


Post-dedup shape: (10080, 48)


### Duplicate Notes

Exact duplicates và duplicate URL trong cùng source được loại bỏ vì đây là trùng bản ghi rõ ràng. Soft duplicate theo spec chỉ được flag, không drop, vì cùng cấu hình laptop có thể xuất hiện nhiều listing/shop hợp lệ với mức giá khác nhau.

## 10. Validation

In [11]:
numeric_canonical_cols = merged[CANONICAL_COLUMNS].select_dtypes(include=[np.number]).columns.tolist()
validation_checks = []

def add_check(check: str, passed: bool, detail: str = "") -> None:
    validation_checks.append({"check": check, "passed": bool(passed), "detail": detail})

add_check("all canonical columns exist", set(CANONICAL_COLUMNS).issubset(merged.columns), str(sorted(set(CANONICAL_COLUMNS) - set(merged.columns))))
add_check("canonical columns are unique", len(CANONICAL_COLUMNS) == len(set(CANONICAL_COLUMNS)))
add_check("source only contains chotot and websach", set(merged["source"].dropna().unique()) <= {"chotot", "websach"}, str(sorted(merged["source"].dropna().unique())))
add_check("each source has more than 0 rows", merged.groupby("source").size().reindex(["chotot", "websach"], fill_value=0).gt(0).all())
add_check("target_price is numeric", pd.api.types.is_numeric_dtype(merged["target_price"]))
add_check("all target_price > 0", merged["target_price"].gt(0).all())
expected_log = np.log1p(merged["target_price"])
add_check("log_target_price finite", np.isfinite(merged["log_target_price"]).all())
add_check("log_target_price equals log1p(target_price)", np.allclose(merged["log_target_price"], expected_log, rtol=1e-10, atol=1e-10, equal_nan=True))
add_check("no infinite values in numeric canonical columns", ~np.isinf(merged[numeric_canonical_cols].to_numpy(dtype=float)).any())
add_check("merged_row_id is unique", merged["merged_row_id"].is_unique)
add_check(
    "merged_row_id is sequential",
    merged["merged_row_id"].equals(pd.Series(np.arange(len(merged)), index=merged.index)),
)
add_check("source_row_id is not missing", merged["source_row_id"].notna().all())
add_check("target_price_source is not missing", merged["target_price_source"].notna().all())

required_new_cols = [
    "cpu_name_raw",
    "cpu_type",
    "cpu_model_clean",
    "gpu_tier_v2",
]
missing_new_cols = [c for c in required_new_cols if c not in merged.columns]
add_check("new CPU/GPU merge columns exist", not missing_new_cols, str(missing_new_cols))
add_check("Chợ Tốt cpu_type is NaN", chotot_can["cpu_type"].isna().all())
add_check("Chợ Tốt cpu_model_clean is NaN", chotot_can["cpu_model_clean"].isna().all())
add_check("Chợ Tốt gpu_tier_v2 is NaN", chotot_can["gpu_tier_v2"].isna().all())

validation_results = pd.DataFrame(validation_checks)
display(validation_results)
assert validation_results["passed"].all(), validation_results.loc[~validation_results["passed"]].to_string(index=False)
assert not missing_new_cols, f"Missing new columns in merged: {missing_new_cols}"
assert chotot_can["cpu_type"].isna().all(), "Expected Chợ Tốt cpu_type to be NaN"
assert chotot_can["cpu_model_clean"].isna().all(), "Expected Chợ Tốt cpu_model_clean to be NaN"
assert chotot_can["gpu_tier_v2"].isna().all(), "Expected Chợ Tốt gpu_tier_v2 to be NaN"
print("New CPU/GPU merge columns validated successfully.")

,check,passed,detail
0,all canonical columns exist,True,[]
1,canonical columns are unique,True,
2,source only contains chotot and websach,True,"['chotot', 'websach']"
3,each source has more than 0 rows,True,
4,target_price is numeric,True,
5,all target_price > 0,True,
6,log_target_price finite,True,
7,log_target_price equals log1p(target_price),True,
8,no infinite values in numeric canonical columns,True,
9,merged_row_id is unique,True,


New CPU/GPU merge columns validated successfully.


## 11. Source Comparison Report

In [12]:
report_rows = []

def add_report_row(section: str, metric: str, value, source=None, group_col=None, group_value=None) -> None:
    report_rows.append({
        "section": section,
        "metric": metric,
        "source": source,
        "group_col": group_col,
        "group_value": group_value,
        "value": value,
    })

for _, row in count_by_source(merged, "rows").iterrows():
    add_report_row("A_row_counts", "rows", int(row["rows"]), source=row["source"])

for _, row in summarize_target_by_source(merged).iterrows():
    for metric in ["count", "mean", "median", "std", "min", "p05", "p25", "p75", "p95", "max"]:
        add_report_row("B_target_summary_by_source", metric, row[metric], source=row["source"])

missing_rate_cols = [
    "ram_gb", "storage_gb", "screen_size_inch",
    "cpu_name_raw", "cpu_type", "cpu_model_clean",
    "cpu_brand", "cpu_tier", "gpu_tier", "gpu_tier_v2",
    "storage_type_clean", "condition_clean", "warranty_status", "origin_clean",
]
for source, group in merged.groupby("source"):
    for col in missing_rate_cols:
        add_report_row("C_missing_like_by_source", "missing_like_rate", missing_like_rate(group[col]), source=source, group_col="column", group_value=col)
        add_report_row("C_missing_like_by_source", "missing_like_count", missing_like_count(group[col]), source=source, group_col="column", group_value=col)

categorical_cardinality_cols = [
    "brand_grouped", "model_grouped",
    "cpu_name_raw", "cpu_type", "cpu_model_clean",
    "cpu_brand", "cpu_tier", "gpu_tier", "gpu_tier_v2",
    "storage_type_clean", "condition_clean",
]
for source, group in merged.groupby("source"):
    for col in categorical_cardinality_cols:
        add_report_row("D_categorical_cardinality_by_source", "n_unique", int(group[col].nunique(dropna=True)), source=source, group_col="column", group_value=col)

for source, group in merged.groupby("source"):
    top_brands = group["brand_grouped"].value_counts(dropna=False).head(20)
    for rank, (brand, rows) in enumerate(top_brands.items(), start=1):
        add_report_row("E_top_brands_by_source", "rows", int(rows), source=source, group_col="brand_grouped", group_value=brand)
        add_report_row("E_top_brands_by_source", "rank", rank, source=source, group_col="brand_grouped", group_value=brand)

for _, row in merged.groupby(["source", "brand_grouped"], dropna=False).agg(rows=("target_price", "size"), median_target_price=("target_price", "median")).reset_index().iterrows():
    add_report_row("F_median_price_by_source_and_brand", "rows", int(row["rows"]), source=row["source"], group_col="brand_grouped", group_value=row["brand_grouped"])
    add_report_row("F_median_price_by_source_and_brand", "median_target_price", row["median_target_price"], source=row["source"], group_col="brand_grouped", group_value=row["brand_grouped"])

for _, row in merged.groupby(["source", "cpu_tier"], dropna=False).agg(rows=("target_price", "size"), median_target_price=("target_price", "median")).reset_index().iterrows():
    add_report_row("G_median_price_by_source_and_cpu_tier", "rows", int(row["rows"]), source=row["source"], group_col="cpu_tier", group_value=row["cpu_tier"])
    add_report_row("G_median_price_by_source_and_cpu_tier", "median_target_price", row["median_target_price"], source=row["source"], group_col="cpu_tier", group_value=row["cpu_tier"])

add_report_row("H_duplicate_diagnostics", "exact_duplicates_removed", exact_duplicate_removed_count)
add_report_row("H_duplicate_diagnostics", "url_duplicates_removed", url_duplicate_removed_count)
add_report_row("H_duplicate_diagnostics", "soft_duplicate_rows", soft_duplicate_count)
add_report_row("H_duplicate_diagnostics", "cross_source_soft_duplicate_rows", cross_source_soft_duplicate_count)

websach_rows = merged[merged["source"].eq("websach")]
websach_price_diagnostics = {
    "price_spread_warn_rows": int(to_bool_series(websach_rows["flag_price_spread_warn"], default=False).sum()),
    "price_spread_critical_rows": int(to_bool_series(websach_rows["flag_price_spread_critical"], default=False).sum()),
    "price_spread_warn_rate": float(to_bool_series(websach_rows["flag_price_spread_warn"], default=False).mean()) if len(websach_rows) else np.nan,
    "price_spread_critical_rate": float(to_bool_series(websach_rows["flag_price_spread_critical"], default=False).mean()) if len(websach_rows) else np.nan,
}
for metric, value in websach_price_diagnostics.items():
    add_report_row("I_websach_price_diagnostics", metric, value, source="websach")

report = pd.DataFrame(report_rows, columns=["section", "metric", "source", "group_col", "group_value", "value"])
display(report.head(40))
print("Report shape:", report.shape)
display(report["section"].value_counts().rename_axis("section").reset_index(name="rows"))

,section,metric,source,group_col,group_value,value
0,A_row_counts,rows,chotot,None,None,"5,698.0000"
1,A_row_counts,rows,websach,None,None,"4,382.0000"
2,B_target_summary_by_source,count,chotot,None,None,"5,698.0000"
3,B_target_summary_by_source,mean,chotot,None,None,"10,078,167.9517"
4,B_target_summary_by_source,median,chotot,None,None,"7,890,000.0000"
5,B_target_summary_by_source,std,chotot,None,None,"7,922,539.1471"
6,B_target_summary_by_source,min,chotot,None,None,"1,000,000.0000"
7,B_target_summary_by_source,p05,chotot,None,None,"1,900,000.0000"
8,B_target_summary_by_source,p25,chotot,None,None,"4,700,000.0000"
9,B_target_summary_by_source,p75,chotot,None,None,"13,000,000.0000"


Report shape: (230, 6)


,section,rows
0,C_missing_like_by_source,56
1,E_top_brands_by_source,48
2,F_median_price_by_source_and_brand,48
3,G_median_price_by_source_and_cpu_tier,26
4,D_categorical_cardinality_by_source,22
5,B_target_summary_by_source,20
6,H_duplicate_diagnostics,4
7,I_websach_price_diagnostics,4
8,A_row_counts,2


### Report Notes

Report được lưu dạng long format để notebook sau có thể filter theo `section`, `metric`, `source`, `group_col`, và `group_value`. Missing-like rate tính cả `Unknown`/`Missing`, nên phản ánh tốt hơn các cột categorical đã được fill giá trị audit.

## 12. Leakage Documentation

In [13]:
SOURCE_LEAKAGE_COLUMNS_BY_SOURCE = {
    "chotot": ["_price", "price", "price_segment", "is_price_missing", "is_price_outlier", "new_low_price"],
    "websach": [
        "shop_1_price", "shop_2_price", "shop_3_price",
        "shop_1_price_clean", "shop_2_price_clean", "shop_3_price_clean",
        "shop_1_price_domain", "shop_2_price_domain", "shop_3_price_domain",
        "price_row_median_domain", "price_median", "price_min_clean", "price_max_clean",
        "log_price_median", "price_segment",
    ],
    "canonical_target": LABEL_COLUMNS,
}

SOURCE_LEAKAGE_COLUMNS = SOURCE_LEAKAGE_COLUMNS_BY_SOURCE["chotot"] + SOURCE_LEAKAGE_COLUMNS_BY_SOURCE["websach"]
leakage_found = sorted(set(SOURCE_LEAKAGE_COLUMNS).intersection(merged.columns))
print("Source-specific leakage columns found in merged dataframe:", leakage_found)
assert not leakage_found, f"Source-specific leakage columns found in merged data: {leakage_found}"

LEAKAGE_NOTES = [
    "Source-specific target/leakage columns are excluded from the merged canonical dataframe.",
    "target_price and log_target_price are labels, not modeling features.",
    "target_price_source is audit-only and should not be used as a baseline feature.",
    "source is retained for audit and later bias/source-split evaluation; it should not be included in the default baseline feature set.",
    "source may be used in a separate source-aware modeling experiment to quantify source bias and deployment assumptions.",
]
for note in LEAKAGE_NOTES:
    print("-", note)

Source-specific leakage columns found in merged dataframe: []
- Source-specific target/leakage columns are excluded from the merged canonical dataframe.
- target_price and log_target_price are labels, not modeling features.
- target_price_source is audit-only and should not be used as a baseline feature.
- source is retained for audit and later bias/source-split evaluation; it should not be included in the default baseline feature set.
- source may be used in a separate source-aware modeling experiment to quantify source bias and deployment assumptions.


## 13. Save Outputs

In [14]:
MERGED_PATH.parent.mkdir(parents=True, exist_ok=True)

merged.to_csv(MERGED_PATH, index=False, encoding="utf-8")
report.to_csv(REPORT_PATH, index=False, encoding="utf-8")

target_summary_for_config = summarize_target_by_source(merged)
merge_config = {
    "input_paths": {"chotot": CHOTOT_PATH, "websach": WEBSACH_PATH},
    "output_paths": {"merged": MERGED_PATH, "report": REPORT_PATH, "config": CONFIG_PATH},
    "canonical_columns": CANONICAL_COLUMNS,
    "label_columns": LABEL_COLUMNS,
    "audit_columns": AUDIT_COLUMNS,
    "feature_exclusion_recommendation": {
        "always_exclude": list(dict.fromkeys(LABEL_COLUMNS + AUDIT_COLUMNS)),
        "optional_exclude": ["source"],
        "notes": [
            "Do not use target_price/log_target_price as features.",
            "Do not include source in baseline unless running a source-aware experiment.",
            "Use target_price_source only for audit.",
        ],
    },
    "source_mappings": {"chotot": CHOTOT_MAPPING, "websach": WEBSACH_MAPPING},
    "defaults": {"chotot": CHOTOT_DEFAULTS, "websach": WEBSACH_DEFAULTS},
    "leakage_columns": SOURCE_LEAKAGE_COLUMNS_BY_SOURCE,
    "source_specific_leakage_columns_flat": SOURCE_LEAKAGE_COLUMNS,
    "leakage_notes": LEAKAGE_NOTES,
    "soft_duplicate_key_columns": SOFT_DUPLICATE_KEY_COLUMNS,
    "row_counts_before_filtering": row_counts_before_filtering,
    "row_counts_after_filtering": row_counts_after_filtering,
    "filter_audit": filter_audit,
    "exact_duplicate_removed_count": exact_duplicate_removed_count,
    "url_duplicate_removed_count": url_duplicate_removed_count,
    "soft_duplicate_count": soft_duplicate_count,
    "cross_source_soft_duplicate_count": cross_source_soft_duplicate_count,
    "websach_price_diagnostics": websach_price_diagnostics,
    "final_shape": merged.shape,
    "target_summary_by_source": target_summary_for_config,
    "validation_results": validation_results,
}

with CONFIG_PATH.open("w", encoding="utf-8") as f:
    json.dump(to_jsonable(merge_config), f, ensure_ascii=False, indent=2)

print("Saved merged CSV:", MERGED_PATH)
print("Saved report CSV:", REPORT_PATH)
print("Saved config JSON:", CONFIG_PATH)

Saved merged CSV: Y:\Python\Laptop-Price-Prediction\data\intern\laptop_merged_cleaned.csv
Saved report CSV: Y:\Python\Laptop-Price-Prediction\docs\laptop_merged_report.csv
Saved config JSON: Y:\Python\Laptop-Price-Prediction\docs\laptop_merge_config.json


## 14. Final Summary

In [15]:
rows_kept = merged.groupby("source").size().reindex(["chotot", "websach"], fill_value=0)
median_price_by_source = merged.groupby("source")["target_price"].median().reindex(["chotot", "websach"])
missing_rates_core = (
    merged.groupby("source")[["ram_gb", "storage_gb", "screen_size_inch"]]
    .apply(lambda g: pd.Series({col: missing_like_rate(g[col]) for col in g.columns}))
    .reindex(["chotot", "websach"])
)

summary = {
    "chotot_rows_kept": int(rows_kept.loc["chotot"]),
    "websach_rows_kept": int(rows_kept.loc["websach"]),
    "total_merged_rows": int(len(merged)),
    "canonical_column_count": int(len(CANONICAL_COLUMNS)),
    "soft_duplicate_count": int(soft_duplicate_count),
    "cross_source_soft_duplicate_count": int(cross_source_soft_duplicate_count),
    "websach_price_diagnostics": websach_price_diagnostics,
    "merged_path": str(MERGED_PATH),
    "report_path": str(REPORT_PATH),
    "config_path": str(CONFIG_PATH),
}

print("Final merge summary")
print(json.dumps(to_jsonable(summary), ensure_ascii=False, indent=2))
print("\nMedian target price by source:")
display(median_price_by_source.rename("median_target_price").reset_index())
print("Missing-like rates for RAM/storage/screen by source:")
display(missing_rates_core)
print("Saved file paths:")
print(MERGED_PATH)
print(REPORT_PATH)
print(CONFIG_PATH)

Final merge summary
{
  "chotot_rows_kept": 5698,
  "websach_rows_kept": 4382,
  "total_merged_rows": 10080,
  "canonical_column_count": 48,
  "soft_duplicate_count": 6882,
  "cross_source_soft_duplicate_count": 0,
  "websach_price_diagnostics": {
    "price_spread_warn_rows": 583,
    "price_spread_critical_rows": 142,
    "price_spread_warn_rate": 0.1330442720219078,
    "price_spread_critical_rate": 0.03240529438612506
  },
  "merged_path": "Y:\\Python\\Laptop-Price-Prediction\\data\\intern\\laptop_merged_cleaned.csv",
  "report_path": "Y:\\Python\\Laptop-Price-Prediction\\docs\\laptop_merged_report.csv",
  "config_path": "Y:\\Python\\Laptop-Price-Prediction\\docs\\laptop_merge_config.json"
}

Median target price by source:


,source,median_target_price
0,chotot,"7,890,000.0000"
1,websach,"21,195,000.0000"


Missing-like rates for RAM/storage/screen by source:


,ram_gb,storage_gb,screen_size_inch
source,,,
chotot,0.0277,0.0618,0.1565
websach,0.0052,0.0513,0.0219


Saved file paths:
Y:\Python\Laptop-Price-Prediction\data\intern\laptop_merged_cleaned.csv
Y:\Python\Laptop-Price-Prediction\docs\laptop_merged_report.csv
Y:\Python\Laptop-Price-Prediction\docs\laptop_merge_config.json


## Decision Summary

- Notebook 04 chỉ merge và canonicalize, chưa làm feature engineering chính thức.
- Target chung là `target_price`.
- Log target chung là `log_target_price`, luôn được recompute từ `target_price`.
- Chợ Tốt outlier price bị loại khỏi merged modeling population.
- Websach price spread flags được giữ làm diagnostics, không dùng để filter.
- Exact duplicates và duplicate URL trong cùng source được drop; cross-source soft duplicates chỉ được flag, không drop.
- `source` được giữ cho audit và source-split evaluation. Không đưa vào baseline feature mặc định; chỉ dùng trong experiment riêng nếu muốn kiểm tra source-aware model.
- Output chính cho notebook sau là `data/intern/laptop_merged_cleaned.csv`.